In [28]:
import sys
from pathlib import Path

week6_root = Path.cwd().resolve().parents[1]  
sys.path.insert(0, str(week6_root))

import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from openai import OpenAI
from pricer.items import Item

import gradio as gr




In [ ]:
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"\nSample item: {test[0]}")
print(f"Summary:\n{test[0].summary}")
print(f"Price: ${test[0].price}")

In [ ]:
prices = [item.price for item in train]
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.hist(prices, bins=range(0, 1000, 10), color="blueviolet", rwidth=0.7)
plt.title(f"Price Distribution (avg ${np.mean(prices):.0f})")
plt.xlabel("Price ($)")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
lengths = [len(item.summary) for item in train]
plt.hist(lengths, bins=50, color="skyblue", rwidth=0.7)
plt.title(f"Summary Length Distribution (avg {np.mean(lengths):.0f} chars)")
plt.xlabel("Length (chars)")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

In [32]:
def evaluate(predictor, data, size=200):
    errors = []
    guesses = []
    truths = []

    for i in tqdm(range(min(size, len(data)))):
        item = data[i]
        try:
            raw = predictor(item)
            if isinstance(raw, str):
                raw = raw.replace("$", "").replace(",", "")
                match = re.search(r"[-+]?\d*\.?\d+", raw)
                guess = float(match.group()) if match else 0
            else:
                guess = float(raw)
        except Exception:
            guess = 0

        truth = item.price
        error = abs(guess - truth)
        errors.append(error)
        guesses.append(guess)
        truths.append(truth)

    mae = np.mean(errors)
    mse = mean_squared_error(truths, guesses)
    r2 = r2_score(truths, guesses) * 100

    print(f"\nResults ({size} items):")
    print(f"  Mean Absolute Error: ${mae:,.2f}")
    print(f"  MSE: {mse:,.0f}")
    print(f"  R²: {r2:.1f}%")

    plt.figure(figsize=(8, 8))
    max_val = max(max(truths), max(guesses))
    colors = ["green" if e < 40 or e / t < 0.2 else "orange" if e < 80 or e / t < 0.4 else "red"
              for e, t in zip(errors, truths)]
    plt.scatter(truths, guesses, c=colors, s=10, alpha=0.7)
    plt.plot([0, max_val], [0, max_val], "b--", alpha=0.5, label="Perfect prediction")
    plt.xlabel("Actual Price ($)")
    plt.ylabel("Predicted Price ($)")
    plt.title(f"MAE: ${mae:.2f} | R²: {r2:.1f}%")
    plt.legend()
    plt.show()

    return mae

In [ ]:
def random_pricer(item):
    return random.randrange(1, 1000)

random.seed(42)
evaluate(random_pricer, test)

In [ ]:
avg_price = np.mean([item.price for item in train])
print(f"Training average price: ${avg_price:.2f}")

def constant_pricer(item):
    return avg_price

evaluate(constant_pricer, test)

In [ ]:
def get_features(item):
    return {
        "weight": item.weight or 0,
        "weight_unknown": 1 if not item.weight else 0,
        "text_length": len(item.summary or ""),
    }

train_df = pd.DataFrame([get_features(item) for item in train])
train_df["price"] = [item.price for item in train]

test_df = pd.DataFrame([get_features(item) for item in test])
test_df["price"] = [item.price for item in test]

feature_cols = ["weight", "weight_unknown", "text_length"]
lr_model = LinearRegression()
lr_model.fit(train_df[feature_cols], train_df["price"])

for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  Intercept: {lr_model.intercept_:.4f}")

def linear_regression_pricer(item):
    features = pd.DataFrame([get_features(item)])
    return lr_model.predict(features)[0]

evaluate(linear_regression_pricer, test)

In [ ]:
documents = [item.summary for item in train]
prices = np.array([item.price for item in train])

vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_train_bow = vectorizer.fit_transform(documents)

bow_regressor = LinearRegression()
bow_regressor.fit(X_train_bow, prices)

def bow_linear_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, bow_regressor.predict(x)[0])

evaluate(bow_linear_pricer, test)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X_train_bow[:15000], prices[:15000])

def random_forest_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

evaluate(random_forest_pricer, test)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X_train_bow, prices)

def xgboost_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

evaluate(xgboost_pricer, test)

In [ ]:
hash_vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X_nn = hash_vectorizer.fit_transform(documents)
X_tensor = torch.FloatTensor(X_nn.toarray())
y_tensor = torch.FloatTensor(prices).unsqueeze(1)

class PriceNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
net = PriceNet(X_tensor.shape[1])
print(f"Parameters: {sum(p.numel() for p in net.parameters()):,}")

loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=64, shuffle=True)
optimizer = optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(2):
    net.train()
    for batch_X, batch_y in tqdm(loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        loss = loss_fn(net(batch_X), batch_y)
        loss.backward()
        optimizer.step()
    net.eval()
    with torch.no_grad():
        val_docs = [item.summary for item in val]
        X_val = torch.FloatTensor(hash_vectorizer.transform(val_docs).toarray())
        y_val = torch.FloatTensor([item.price for item in val]).unsqueeze(1)
        val_loss = loss_fn(net(X_val), y_val)
    print(f"  Val Loss: {val_loss.item():.2f}")

def neural_network_pricer(item):
    net.eval()
    with torch.no_grad():
        x = torch.FloatTensor(hash_vectorizer.transform([item.summary]).toarray())
        return max(0, net(x).item())

evaluate(neural_network_pricer, test)

In [ ]:
openai = OpenAI()

def gpt_nano_pricer(item):
    response = openai.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}],
        max_tokens=10,
    )
    return response.choices[0].message.content

evaluate(gpt_nano_pricer, test, size=50)

In [ ]:
def make_ft_messages(item):
    return {
        "messages": [
            {"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"},
            {"role": "assistant", "content": f"${item.price:.2f}"},
        ]
    }

ft_train = train[:100]
ft_val = val[:50]

with open("ft_train.jsonl", "w") as f:
    for item in ft_train:
        f.write(json.dumps(make_ft_messages(item)) + "\n")

with open("ft_val.jsonl", "w") as f:
    for item in ft_val:
        f.write(json.dumps(make_ft_messages(item)) + "\n")

with open("ft_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("ft_val.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="valoni-pricer",
)
print(f"Fine-tuning job started: {job.id}")

In [ ]:
status = openai.fine_tuning.jobs.retrieve(job.id)
print(f"Status: {status.status}")
print(f"Model: {status.fine_tuned_model}")

In [ ]:
ft_model_name = openai.fine_tuning.jobs.retrieve(job.id).fine_tuned_model

def gpt_nano_finetuned_pricer(item):
    response = openai.chat.completions.create(
        model=ft_model_name,
        messages=[{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}],
        max_tokens=7,
    )
    return response.choices[0].message.content

evaluate(gpt_nano_finetuned_pricer, test)

In [27]:

def predict_all(description):
    """Run all models on a product description and compare results."""

    class FakeItem:
        def __init__(self, summary):
            self.summary = summary
            self.weight = 0
            self.title = summary[:50]
            self.price = 0

    item = FakeItem(description)
    results = {}

    results["Constant (avg)"] = f"${avg_price:.2f}"

    lr_pred = linear_regression_pricer(item)
    results["Linear Regression"] = f"${lr_pred:.2f}"

    bow_pred = bow_linear_pricer(item)
    results["BoW Linear"] = f"${bow_pred:.2f}"

    rf_pred = random_forest_pricer(item)
    results["Random Forest"] = f"${rf_pred:.2f}"

    xgb_pred = xgboost_pricer(item)
    results["XGBoost"] = f"${xgb_pred:.2f}"

    nn_pred = neural_network_pricer(item)
    results["Neural Network"] = f"${nn_pred:.2f}"

    try:
        gpt_pred = gpt_nano_pricer(item)
        results["GPT-4.1-nano (zero-shot)"] = gpt_pred
    except Exception as e:
        results["GPT-4.1-nano (zero-shot)"] = f"Error: {e}"

    try:
        ft_pred = gpt_nano_finetuned_pricer(item)
        results["GPT-4.1-nano (fine-tuned)"] = ft_pred
    except Exception:
        results["GPT-4.1-nano (fine-tuned)"] = "Not available yet"

    output = "\n".join(f"{model:.<35} {price}" for model, price in results.items())
    return output

demo = gr.Interface(
    fn=predict_all,
    inputs=gr.Textbox(
        lines=6,
        placeholder="Title: Sony WH-1000XM5 Headphones\nCategory: Electronics\nBrand: Sony\nDescription: Premium wireless noise-canceling headphones.\nDetails: 30-hour battery, multipoint connection, adaptive sound control.",
        label="Product Description",
    ),
    outputs=gr.Textbox(label="Price Predictions", lines=12),
    title="The Price Is Right - Model Comparison",
    description="Enter a product description and see how every model predicts the price.",
    flagging_mode="never",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
